[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/r1-q1/03_consensus_compare.ipynb)

# R1-Q1 Week 4 — Consensus Comparison & Report

You're in Week 4 of R1-Q1. Last week you identified the common stress core — 9,067 genes differentially expressed in at least 6 of the 8 abiotic stresses in AtGenExpress — and ran GO over-representation analysis to characterize what those genes do functionally. This week takes that core and compares it to a published meta-analytic consensus: Hakkak & Tohidfar 2026's signature of conserved drought and salt response in *Arabidopsis thaliana*. The comparison closes the R1-Q1 arc and produces the material for the final paper.

**By the end of this notebook you will have:**

- The core gene set translated from ATH1 probe IDs to AGI locus codes.
- A pre-committed prediction of how the two cores should overlap, written down before seeing the data.
- The overlap result — shared genes, ours-only, theirs-only — with hypergeometric significance, plus a three-region breakdown checked against the prediction.
- Targeted comparisons against Hakkak & Tohidfar's transcription factor list and the specific hub genes named in their paper.
- A synthesis pulling the comparison together with the enrichment findings from Notebook 2, and a saved comparison artifact for the final paper.

## 1) Setup and data

Install the iRI Lab library, mount Drive, read the common stress core that Notebook 2 wrote to Drive, and sanity-check what loaded.

In [ ]:
# Setup. Four steps: install the iRI Lab library, mount Drive,
# read the common stress core that last week's notebook wrote to Drive,
# and sanity-check what loaded.

# 1. Install the library (not pre-installed on Colab).
!pip install --upgrade --no-cache-dir git+https://github.com/geraldmc/irilab2026.git@main --quiet

In [ ]:
# Setup Step 2:  Mount Drive, read the common stress core, and sanity-check what loaded.

# 2. Mount Drive.
import irilab2026 as iri
iri.setup()

# 3. Read the common stress core from Drive — written by 02_core_overlap.ipynb.
import pandas as pd
core_genes = pd.read_parquet(iri.output_dir("r1_q1") / "core_genes.parquet")

# 4. Sanity-check what loaded.
print(f"Loaded: {core_genes.shape}")
print(f"n_stresses distribution:\n{core_genes['n_stresses'].value_counts().sort_index()}")
core_genes.head()

### 1.1) Inputs this notebook needs

Two files:

1. **`core_genes.parquet`** — produced by Notebook 2, sitting in your Drive at `irilab2026_outputs/r1_q1/`. The cell above confirms it is accessible.

2. **Hakkak & Tohidfar 2026 Supplementary File 1** — the meta-analytic DEG list from the published paper. We load this in Section 4. Download instructions and the file naming wrinkle are in that section.

## 2) Translate probes to AGI

The core we just loaded is keyed on ATH1 probe IDs (e.g., `244903_at`). Hakkak & Tohidfar's consensus list is keyed on AGI codes (e.g., `At5g59310`) — the standard *Arabidopsis* Genome Initiative identifiers used in nearly every gene-level resource. We can't compare the two lists directly until we translate one of them. We translate ours.

We also need a **background** — the full set of genes that *could* have ended up in the core. The background is every probe that appeared in last week's DE analysis, whether it was flagged DE or not. We need it because Section 4 asks "is the overlap with Hakkak's list bigger than chance?", and answering that requires knowing the universe the core was drawn from. We translate the background to AGI too, so the comparison runs on a common identifier system.

The translation uses the GPL198 annotation table — GEO's accession for the Affymetrix ATH1 platform, which maps every probe ID on the chip to its target AGI locus. Two defensive cases come up. **Multi-locus probes** target more than one AGI locus, listed as `At1g01010 /// At1g01020`; we take the first locus, which is the probe's primary target. **Probes without AGI** can't participate in the comparison and get dropped — this is also where the *Affymetrix spike-in control probes* you saw in the core (prefixed `AFFX-`, bacterial sequences used for hybridization quality control) drop out, since they aren't *Arabidopsis* genes and have no AGI codes.

In [ ]:
# Load the long-format DE table from Notebook 2. We need every probe that
# appeared in the DE analysis — that's the universe for the hypergeometric
# test in Section 4.
de_results = pd.read_parquet(iri.output_dir("r1_q1") / "de_results.parquet")

print(f"DE results loaded: {de_results.shape}")
print(f"Unique probes:     {de_results['gene'].nunique():,}")

In [ ]:
# Fetch the GPL198 annotation table. GEOparse caches the SOFT file in
# iri.cache_dir() so a re-run doesn't hit the network a second time.
import GEOparse

gpl = GEOparse.get_GEO(geo='GPL198', destdir=str(iri.cache_dir()), silent=True)

print(f"Annotation table: {gpl.table.shape}")
gpl.table[['ID', 'AGI']].head()

In [ ]:
# Build a probe -> AGI mapping from the GPL198 annotation table.
#
# Two cases to handle defensively:
#   1. Multi-locus probes: a probe may target multiple AGI loci, separated
#      by ' /// '. We take the first locus (the probe's primary target).
#   2. Missing AGI: some probes have no AGI annotation at all (empty or NaN).
#      Those probes can't participate in the comparison and get dropped.
#      This is where AFFX control probes get filtered out.
probe_to_agi = {}
for _, row in gpl.table[['ID', 'AGI']].dropna().iterrows():
    probe = row['ID']
    first_locus = str(row['AGI']).split(' /// ')[0].strip().upper()
    if first_locus:
        probe_to_agi[probe] = first_locus

print(f"Probes with AGI mapping:  {len(probe_to_agi):,}")
print(f"Probes without AGI:       {len(gpl.table) - len(probe_to_agi):,}")

# Translate the core (9,067 probes -> AGI codes).
core_agi = (
    core_genes['gene']
    .map(probe_to_agi)
    .dropna()
    .unique()
    .tolist()
)

# Translate the background — every probe we ran DE on.
background_agi = (
    de_results['gene']
    .drop_duplicates()
    .map(probe_to_agi)
    .dropna()
    .unique()
    .tolist()
)

print()
print(f"Core (AGI):       {len(core_agi):,} unique loci")
print(f"Background (AGI): {len(background_agi):,} unique loci")

## 3) Pre-committed prediction

Before loading Hakkak & Tohidfar's data, we write down what we expect the comparison to show. Same pattern as Notebook 2's N=6 supermajority commitment: decide the rules in advance, in writing, so the result doesn't reduce to "whichever interpretation makes the answer look cleanest."

A comparison between two gene lists has three regions. **The intersection** — genes in both lists. **Theirs-only** — genes in Hakkak's consensus that aren't in our core. **Ours-only** — genes in our core that aren't in Hakkak's consensus. Each region gets its own prediction about what biology should dominate it.

The framing matters because the comparison isn't apples-to-apples. Our core spans eight abiotic stresses and counts a gene if it's DE in at least six. Hakkak's consensus is built from a meta-analysis of drought-and-salt-only studies. The overlap should be the cross-tolerant subset of drought/salt response by construction — genes that respond not just to drought and salt but to most of our other six stresses as well. The non-overlapping regions should each pick up biology specific to the broader-versus-narrower stress set.

The predictions, written down before we look:

- **Intersection.** Shared cross-tolerance machinery — broad ROS (reactive oxygen species) metabolism, generic stress-responsive TFs like WRKY and NAC, HSPs (heat shock proteins) that act across multiple proteostasis stresses. The backbone of stress response, not drought/salt-specific biology.
- **Theirs-only.** Drought/salt-specific biology — ion homeostasis (the SOS pathway, Na⁺/K⁺ transporters), water-deprivation specifics (LEA proteins, dehydrins), osmolyte biosynthesis, ABA-specific signaling. Genes whose response is concentrated in water/ion stress rather than spread across many stresses.
- **Ours-only.** Biology shared across our eight stresses but not specifically drought/salt — hypoxia/decreased-oxygen response (Notebook 2's top enrichment cluster), and wound/genotoxic/UV-specific machinery that recurs across our other stresses.

A result that would genuinely surprise us: if Hakkak's drought/salt-specific genes (ion transporters, LEA proteins, ABA-specific components) appear substantially in our core. That would contradict the textbook view that ion homeostasis is salt-specific and water-deprivation machinery is drought-specific. We note this as the falsification anchor — if it shows up, the prediction was wrong in a substantive way and Section 7's synthesis has to account for it.

The cell below captures the prediction as a Python dict so Section 4's three-region breakdown can reference it explicitly. Pre-committed means pre-committed — the dict gets written once, here.

In [ ]:
# The three-region prediction, captured as a dict so Section 4 can
# reference it explicitly rather than re-stating it. Pre-committed
# means committed in writing before we look at Hakkak's data.
prediction = {
    "intersection": (
        "Shared cross-tolerance machinery — broad ROS metabolism, "
        "generic stress-responsive TFs (WRKY, NAC), HSPs."
    ),
    "theirs_only": (
        "Drought/salt-specific biology — ion homeostasis (SOS pathway, "
        "Na+/K+ transporters), LEA proteins, dehydrins, osmolyte "
        "biosynthesis, ABA-specific signaling."
    ),
    "ours_only": (
        "Cross-stress but not drought/salt-specific — hypoxia response "
        "(Notebook 2's top enrichment cluster), and wound/genotoxic/UV "
        "machinery that recurs across our other stresses."
    ),
    "would_surprise": (
        "Drought/salt-specific genes appearing substantially in our core — "
        "would contradict the canonical view that ion homeostasis is "
        "salt-specific and water-deprivation machinery is drought-specific."
    ),
}

for region, text in prediction.items():
    label = region.upper().replace("_", " ")
    print(f"\n{label}:")
    print(f"  {text}")

### 3.1) Load Hakkak & Tohidfar's consensus

The CSV file at `irilab2026_outputs/r1_q1/hakkak_2026_supp1.csv` (on Google Drive) holds the differentially expressed genes that Hakkak & Tohidfar identified through their meta-analysis — 397 up-regulated under drought-or-salt and 170 down-regulated. This is the list that we will compare our core to in Section 4.

The file has an unusual layout. Row 1 is a title. Row 2 is the column header. Below the header, the data is split into two side-by-side blocks: columns 1–8 hold the up-regulated genes, then two empty separator columns, then columns 11–18 hold the down-regulated genes. The blocks have different lengths (397 rows vs. 170 rows), so the down-regulated side runs out partway down with empty cells. We need to read both blocks and stack them into a single long-format DataFrame.

One acknowledgment before we start. The paper's abstract and methods both report **576** total DEGs. The supplementary file contains **567** (397 + 170). The 9-gene difference isn't explained in the paper text — likely a transcription discrepancy between manuscript and supplementary that didn't get caught in proofing. The supplementary is the authoritative source, since it's the file the comparison actually uses. We work with 567.

To make sure the file loaded correctly, we validate against two anchors that the paper text names by gene: **At5g59310 (LTP4)** should be the top up-regulated gene at log2FC ≈ 4.46, and **At1g22690 (GASA9)** should be the top down-regulated gene at log2FC ≈ -2.56. If both check out, the file is what the paper says it is and we proceed.

In [ ]:
# Load Hakkak & Tohidfar's primary DEG list. The CSV has an unusual
# layout: title row, header row, then two side-by-side blocks of 8
# columns separated by two empty columns. Up-regulated genes on the
# left, down-regulated on the right.
hakkak_path = iri.output_dir("r1_q1") / "hakkak_2026_supp1.csv"
raw = pd.read_csv(hakkak_path, header=1)

# Split into the two parallel blocks and rename to consistent columns.
cols = ["Gene", "Up_Down", "log2FC", "AveExpr", "t", "P_Value", "adj_P_Val", "B"]
up = raw.iloc[:, 0:8].copy()
down = raw.iloc[:, 10:18].copy()
up.columns = cols
down.columns = cols

# The down-regulated block is shorter (170 vs 397), so drop empty rows.
up = up.dropna(subset=["Gene"]).reset_index(drop=True)
down = down.dropna(subset=["Gene"]).reset_index(drop=True)

# Stack the two blocks and uppercase AGI codes to match Notebook 2's
# probe_to_agi convention (which uppercased on the way in).
hakkak = pd.concat([up, down], ignore_index=True)
hakkak["Gene"] = hakkak["Gene"].str.upper()

print(f"Up-regulated:   {len(up)}")
print(f"Down-regulated: {len(down)}")
print(f"Total unique:   {hakkak['Gene'].nunique()}")
hakkak.head()

In [ ]:
# Validate against the anchors named in the paper text:
#   At5g59310 (LTP4)  — top up-regulated, log2FC ≈ 4.46
#   At1g22690 (GASA9) — top down-regulated, log2FC ≈ -2.56
top_up = up.sort_values("log2FC", ascending=False).iloc[0]
top_down = down.sort_values("log2FC", ascending=True).iloc[0]

# Note: 'up' and 'down' here are still in their original case (At...);
# we only uppercased after concatenation. Uppercase for the comparison.
top_up_gene = top_up['Gene'].upper()
top_down_gene = top_down['Gene'].upper()

print(f"Top up-regulated:   {top_up_gene:12s} log2FC={top_up['log2FC']:.3f}  (paper: AT5G59310, 4.46)")
print(f"Top down-regulated: {top_down_gene:12s} log2FC={top_down['log2FC']:.3f}  (paper: AT1G22690, -2.56)")

assert top_up_gene == 'AT5G59310', f"Expected AT5G59310 as top up, got {top_up_gene}"
assert top_down_gene == 'AT1G22690', f"Expected AT1G22690 as top down, got {top_down_gene}"
print("\nAnchors check out.")

# The list of AGI codes that Section 4 will compare against our core_agi.
hakkak_agi = hakkak["Gene"].unique().tolist()
print(f"\nhakkak_agi: {len(hakkak_agi):,} unique AGI codes")

## 4) Overlap analysis and three-region breakdown

This is the methodological centerpiece of the notebook. We take our core (translated to AGI in Section 1) and Hakkak & Tohidfar's consensus (loaded in Section 3), compute the three regions of overlap, and check the results against the prediction Section 2 pre-committed to.

Three numbers do most of the work. **The intersection** is the count of AGI codes that appear in both lists. **Theirs-only** is the count in Hakkak's list that aren't in our core. **Ours-only** is the count in our core that aren't in Hakkak's list. The three numbers, plus a total, fully describe how the lists relate.

To check whether the overlap is bigger than chance, we run a **hypergeometric test**. The intuition: imagine the background as an urn full of marbles, with Hakkak's genes colored red and the rest colored blue. We draw a number of marbles equal to our core size and count how many are red. The hypergeometric distribution tells us the probability of getting at least that many reds by random draw. A very small p-value means the overlap is much larger than chance — biology, not coincidence.

We use the full `background_agi` (~22,000 testable AGI codes) as the universe. A more conservative version of the test would restrict the universe to AGI codes detectable on both Hakkak's and our platforms; that would shift the p-value modestly but not change the headline finding when the overlap is far from random.

The section ends with a side-by-side display of each region's count and the pre-committed prediction for what biology it should contain. The interpretation — did the prediction hold, what biology actually dominates each region — is the work of Sections 5 through 7. This section produces the numbers and leaves the reading for later.

### 4.1) Computing the three regions

Three set operations give us the gene counts for each Venn region. We convert each list to a Python set — sets dedupe automatically and support overlap arithmetic with the `&` (intersection) and `-` (difference) operators — then compute the intersection, theirs-only, and ours-only. The percentages at the bottom tell us recovery in each direction: what fraction of Hakkak's list ended up in our core, and what fraction of our core is in Hakkak's consensus.

In [ ]:
# Compute the three regions. Sets are the right data structure for
# overlap arithmetic — order doesn't matter, dedup is automatic.
core_set       = set(core_agi)
hakkak_set     = set(hakkak_agi)
background_set = set(background_agi)

intersection = core_set & hakkak_set
ours_only    = core_set - hakkak_set
theirs_only  = hakkak_set - core_set

n_intersection = len(intersection)
n_ours_only    = len(ours_only)
n_theirs_only  = len(theirs_only)

print(f"Our core:           {len(core_set):,} AGI codes")
print(f"Hakkak consensus:   {len(hakkak_set):,} AGI codes")
print(f"Background (universe): {len(background_set):,} AGI codes")
print()
print(f"Intersection:       {n_intersection:,} genes")
print(f"Theirs-only:        {n_theirs_only:,} genes")
print(f"Ours-only:          {n_ours_only:,} genes")
print()
print(f"Of Hakkak's {len(hakkak_set)} genes, {n_intersection} ({100*n_intersection/len(hakkak_set):.1f}%) are in our core.")
print(f"Of our {len(core_set)} core genes, {n_intersection} ({100*n_intersection/len(core_set):.1f}%) are in Hakkak's consensus.")

### 4.2) Hypergeometric significance test

The hypergeometric test answers a single question: given a universe of `N` genes, of which `K` are in Hakkak's list, if we draw a sample of `n` (our core), how likely are we to see at least `k` of them in Hakkak's list by chance alone? A very small p-value means the observed overlap is far larger than chance — biology rather than coincidence.

We use the full `background_agi` as the universe. A more conservative version of the test would restrict the universe to AGI codes detectable on both Hakkak's and our platforms; that would shift the p-value modestly but not change the headline finding when the overlap is far from random.

In [ ]:
# Hypergeometric test: is the overlap larger than expected by chance?
#
# Setup:
#   N = background size (universe of testable AGI codes)
#   K = Hakkak genes that are also in our background (the "successes" present in the universe)
#   n = our core size (the sample drawn)
#   k = observed overlap
from scipy.stats import hypergeom

N = len(background_set)
K = len(hakkak_set & background_set)
n = len(core_set)
k = n_intersection

# Expected overlap under random draw = n * K / N
expected = n * K / N

# P(X >= k) — survival function at k-1
p_value = hypergeom.sf(k - 1, N, K, n)

print(f"Background size (N):             {N:,}")
print(f"Hakkak genes in background (K):  {K:,}")
print(f"Our core size (n):               {n:,}")
print(f"Observed overlap (k):            {k:,}")
print()
print(f"Expected overlap under chance:   {expected:.1f}")
print(f"Fold enrichment over expected:   {k / expected:.2f}×")
print(f"Hypergeometric p-value:          {p_value:.3e}")

### 4.3) Visualizing the overlap

A two-set Venn diagram is the right visual here — three regions, three numbers. The `matplotlib_venn` package handles the proportional circles automatically. We install it at point of use rather than bundling it in `irilab2026`, since only this notebook needs it.

The asymmetry of the diagram will be the main thing worth noticing: Hakkak's small circle should be almost entirely contained within our larger one, with the recovery percentage from 4.1 reflected in how much of their circle gets eaten by the intersection.

In [ ]:
# Render a 2-set Venn diagram. matplotlib_venn isn't bundled with the
# library because only this notebook needs it — install at point of use.
!pip install matplotlib_venn --quiet

from matplotlib_venn import venn2
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 6))
venn2(
    subsets=(n_ours_only, n_theirs_only, n_intersection),
    set_labels=("Our core\n(8 stresses)", "Hakkak & Tohidfar\n(drought + salt)"),
    ax=ax,
)
ax.set_title(
    f"AtGenExpress single-dataset core vs. Hakkak & Tohidfar 2026 consensus\n"
    f"Hypergeometric p = {p_value:.2e}, {k / expected:.1f}× enrichment over chance"
)
plt.tight_layout()
plt.show()

### 4.4) Comparing the result to the pre-committed prediction

The three regions have their counts. The pre-committed prediction from Section 3 said what kind of biology should dominate each region. Below we print them side-by-side — region count on top, predicted biology below — plus the falsification anchor at the end.

This section produces the numbers and the prediction text together. Whether the prediction actually *held* — whether the genes in each region are biologically what we said they should be — is the work of Sections 5, 6, and 7. Section 5 looks at transcription factors specifically. Section 6 spot-checks the hub genes Hakkak named by name. Section 7 synthesizes.

In [ ]:
# Side-by-side: each region's count next to the pre-committed prediction.
# The prediction dict was set in Section 2 and hasn't been touched since.
results = {
    "intersection": n_intersection,
    "theirs_only":  n_theirs_only,
    "ours_only":    n_ours_only,
}

for region, count in results.items():
    label = region.upper().replace("_", " ")
    print(f"\n{label} — {count:,} genes")
    print(f"Predicted: {prediction[region]}")

print(f"\n{'-' * 60}")
print("FALSIFICATION ANCHOR")
print(f"Predicted: {prediction['would_surprise']}")
print("\nWhether this happened is the question Section 6 and Section 7 answer.")